# 07 — Failure Analysis: False Positives, False Negatives, Hard Cases

**Author:** Sherif Abd El-Rady

**Objective:** Understand *when* PatchCore fails. Identify false positives (normal images scored as defective) and false negatives (defect images that slip through), and explain why each category is difficult.

Understanding failure modes is what separates an engineer from someone who just runs a benchmark.


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import FeatureExtractor, MemoryBank, AnomalyEvaluator
from defectdetect.scorer import AnomalyScorer

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC = ROOT / 'data' / 'synthetic'; CATEGORY = ''
    print('Using synthetic dataset')

base     = MVTEC / CATEGORY if CATEGORY else MVTEC
test_dir = base / 'test'
BANK_DIR = ROOT / 'reports' / 'memory_banks'
print('Ready')

In [ ]:
# ── Re-run scoring (from Notebook 05 results) ─────────────────────────────────
bank      = MemoryBank.load(sorted(BANK_DIR.glob('*.pkl'))[-1])
extractor = FeatureExtractor(layers=['layer2','layer3'], input_size=224, device=DEVICE)
scorer    = AnomalyScorer(image_size=(224, 224), gaussian_sigma=4.0)

all_scores, all_labels, all_amaps, all_paths = [], [], [], []
for cls_dir in sorted(test_dir.iterdir()):
    if not cls_dir.is_dir(): continue
    label = 0 if cls_dir.name == 'good' else 1
    for p in sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg')):
        img = cv2.imread(str(p))
        if img is None: continue
        batch = extractor.preprocess([img])
        patches, sp = extractor.extract_patches(batch)
        ps = bank.score(patches, k=1)
        s, m = scorer.compute(ps, sp, n_images=1)
        all_scores.append(float(s[0])); all_labels.append(label)
        all_amaps.append(m[0]); all_paths.append(p)

scores = np.array(all_scores); labels = np.array(all_labels)
amaps  = scorer.normalise(np.stack(all_amaps))

# Find optimal threshold
ev  = AnomalyEvaluator().fit(scores, labels)
met = ev.image_level_metrics()
THR = met['best_threshold']
preds = (scores >= THR).astype(int)

print(f'ROC-AUC={met["roc_auc"]:.4f}  threshold={THR:.6f}')
print(f'TP={((preds==1)&(labels==1)).sum()}  FP={((preds==1)&(labels==0)).sum()}  '
      f'TN={((preds==0)&(labels==0)).sum()}  FN={((preds==0)&(labels==1)).sum()}')

In [ ]:
# ── 7.1  False positives — normal images flagged as defective ─────────────────
fp_idx = np.where((preds == 1) & (labels == 0))[0]
n_show = min(3, len(fp_idx))
print(f'False positives: {len(fp_idx)}')

if n_show > 0:
    fig, axes = plt.subplots(n_show, 3, figsize=(13, n_show * 3.5))
    if n_show == 1: axes = [axes]
    for row, idx in enumerate(fp_idx[:n_show]):
        img_bgr = cv2.imread(str(all_paths[idx]))
        overlay = scorer.overlay(img_bgr, amaps[idx], alpha=0.5)
        axes[row][0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title(f'FP: {all_paths[idx].parent.name}\nscore={scores[idx]:.4f}', fontsize=7)
        axes[row][0].axis('off')
        axes[row][1].imshow(amaps[idx], cmap='hot')
        axes[row][1].set_title('Anomaly map'); axes[row][1].axis('off')
        axes[row][2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[row][2].set_title('Overlay'); axes[row][2].axis('off')
    plt.suptitle('False Positives — Normal images incorrectly flagged', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('No false positives at this threshold.')

In [ ]:
# ── 7.2  False negatives — defect images missed ───────────────────────────────
fn_idx = np.where((preds == 0) & (labels == 1))[0]
n_show = min(3, len(fn_idx))
print(f'False negatives: {len(fn_idx)}')

if n_show > 0:
    fig, axes = plt.subplots(n_show, 3, figsize=(13, n_show * 3.5))
    if n_show == 1: axes = [axes]
    for row, idx in enumerate(fn_idx[:n_show]):
        img_bgr = cv2.imread(str(all_paths[idx]))
        overlay = scorer.overlay(img_bgr, amaps[idx], alpha=0.5)
        axes[row][0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title(f'FN: {all_paths[idx].parent.name}\nscore={scores[idx]:.4f}', fontsize=7)
        axes[row][0].axis('off')
        axes[row][1].imshow(amaps[idx], cmap='hot')
        axes[row][1].set_title('Anomaly map (below threshold)'); axes[row][1].axis('off')
        axes[row][2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[row][2].set_title('Overlay'); axes[row][2].axis('off')
    plt.suptitle('False Negatives — Defect images that slipped through', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('No false negatives at this threshold.')

In [ ]:
# ── 7.3  Score-threshold sensitivity ──────────────────────────────────────────
# Show how FP/FN rates change as we vary the threshold
from sklearn.metrics import confusion_matrix

thresholds = np.percentile(scores, np.arange(0, 101, 5))
fp_rates, fn_rates = [], []

for thr in thresholds:
    p = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, p, labels=[0,1]).ravel()
    fp_rates.append(fp / (fp + tn + 1e-8))
    fn_rates.append(fn / (fn + tp + 1e-8))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, fp_rates, 'o-', color='#e74c3c', label='False Positive Rate', linewidth=2)
ax.plot(thresholds, fn_rates, 's-', color='#3498db', label='False Negative Rate', linewidth=2)
ax.axvline(THR, color='gray', linestyle='--', label=f'Optimal threshold ({THR:.4f})')
ax.set_xlabel('Anomaly score threshold')
ax.set_ylabel('Rate')
ax.set_title('FP/FN Rate vs Threshold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Engineering insight: lower threshold → catch more defects (↓FN) but flag more normal images (↑FP).')
print('The optimal threshold balances the cost of missing defects vs false alarms.')

## Failure Mode Taxonomy

| Failure | Root Cause | Mitigation |
|---|---|---|
| FP — unusual normal | High within-class variation not covered by memory bank | Increase sampling ratio; collect more diverse normals |
| FN — subtle defect | Defect smaller than receptive field (~67px at layer3) | Use layer1 features; reduce input stride |
| FN — similar texture | Defect texture resembles normal patches | Fine-tune backbone on few-shot defect examples |
| FN — uniform defect | Defect occupies most of image — max score = edge only | Use mean or percentile score instead of max |

## Next step

Open `08_production_pipeline.ipynb` — end-to-end production inference script.
